# Compare best simulations with observations

In this notebook, we compare the simulated population from the best-inferred parameters with the observed population and produce Figures 3 and 8 of Ronchi et al. (2026). We also estimates the birth rate for each simulated survey (see Eqs. 20 and 21 in Ronchi et al. 2026) and count the expected number of young magnetars and XDINS-like neutron stars from the simulations.

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognormal_dip-tor_heavy/experiments_paper.zip`, copy it into the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026` and unpack the file so that the results data will be saved in the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper`.

In [ ]:
# import libraries
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import os

import mlpoppyns.simulator.basics.constants as const
from mlpoppyns.simulator.config_simulator import cfg
import utilities.plot_settings

from utilities.load_catalogs import (
    load_atnf_meerkat_catalog,
    load_xray_catalog,
)

from matplotlib.ticker import FuncFormatter

formatter = FuncFormatter(lambda y, _: "{:.16g}".format(y))
from matplotlib.ticker import LogFormatterMathtext

from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde, ks_2samp

Define some spin-down parameters and functions to plot lines of constant magnetic field and spin-down power in the $P-\dot{P}$ diagram.

In [ ]:
# Dimensionless coefficients k_0, k_1, k_2 for a force-free magnetosphere
# taken from Spitkovsky (2006) and Philippov et al. (2014).
# For comparison, in vacuum k_0 = 0 and k_1 = k_2 = 2/3.
k_coefficients = cfg["k_coefficients"]

# Canonical neutron star moment of inertia in [g cm^2] assuming a perfect solid sphere.
NS_inertia = 2.0 / 5.0 * cfg["NS_mass"] * cfg["NS_radius"] ** 2

# Auxiliary quantity beta as defined in Eq. (72) of Pons & Vigano (2019).
beta = np.pi**2 * cfg["NS_radius"] ** 6 / (NS_inertia * const.C**3)

# Assume an inclination angle in [rad].
chi = 0.0

beta_1 = beta * (k_coefficients[0] + k_coefficients[1] * np.sin(chi) ** 2)

In [ ]:
def B_from_timing(P: float, Pdot: float) -> float:
    """
    B field estimated from timing properties.

    Args:
        P (float): Spin period of a simulated pulsar, measured in [s].
        Pdot (float): Period derivative of a simulated pulsar in [s/s].

    Returns:
        (float): Value of the dipolar component of the magnetic field at the
            magnetic pole for a simulated neutron star, measured in [G].
    """

    # Period derivative.
    B = np.sqrt(P * Pdot / beta_1)

    return B


def tage_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar characteristic age from timing properties assuming P >> P0 and constant magnetic field.

    Args:
        P (float): Spin period of a simulated pulsar, measured in [s].
        Pdot (float): Period derivative of a simulated pulsar in [s/s].

    Returns:
        (float): Characteristic age in [yr].
    """

    # Characteristic age definition.
    tage = P / (2 * Pdot) / const.YR_TO_S

    return tage


def Edot_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar rotational power from timing properties.

    Args:
        P (float): Spin period of a simulated pulsar, measured in [s].
        Pdot (float): Period derivative of a simulated pulsar in [s/s].

    Returns:
        (float): Rotational power [erg/s].
    """

    # E_dot definition.
    Edot = 4 * np.pi**2 * NS_inertia * Pdot / P**3

    return Edot

## Load the observed catalogs

In [ ]:
surveys_atnf, surveys_meerkat = load_atnf_meerkat_catalog(
    "../../data/observations/atnf_full_nobinary_25-03-2025_with_errors.csv",
    "../../data/observations/meerkat_tpa_posselt_2023.csv",
)

In [ ]:
P_pmps_obs = surveys_atnf["PMPS"]["P"]
Pdot_pmps_obs = surveys_atnf["PMPS"]["P_dot"]
S1400_pmps_obs = surveys_meerkat["PMPS"]["S1400"] * 1.e3 # convert from Jy to mJy.

P_smps_obs = surveys_atnf["SMPS"]["P"]
Pdot_smps_obs = surveys_atnf["SMPS"]["P_dot"]
S1400_smps_obs = surveys_meerkat["SMPS"]["S1400"] * 1.e3 # convert from Jy to mJy.

P_htru_obs = surveys_atnf["HTRU_low-mid"]["P"]
Pdot_htru_obs = surveys_atnf["HTRU_low-mid"]["P_dot"]
S1400_htru_obs = surveys_meerkat["HTRU_low-mid"]["S1400"] * 1.e3 # convert from Jy to mJy.

In [ ]:
P_radio_obs = np.concatenate((P_pmps_obs, P_smps_obs, P_htru_obs))
Pdot_radio_obs = np.concatenate((Pdot_pmps_obs, Pdot_smps_obs, Pdot_htru_obs))
S_radio_obs = np.concatenate((S1400_pmps_obs, S1400_smps_obs, S1400_htru_obs))

In [ ]:
surveys_xray = load_xray_catalog(
    "../../data/observations/thermal_NS_05-11-2024.csv",
)

In [ ]:
P_x_obs = surveys_xray["P"]
Pdot_x_obs = surveys_xray["P_dot"]
S_x_obs = surveys_xray["S_x_abs"]
age_x_real_obs = surveys_xray["age"]
d_x_real_obs = surveys_xray["dist"]

# Compute the characteristic age in kyr.
age_char_x_obs = tage_from_timing(P_x_obs, Pdot_x_obs) / 1000

# Define the filters for the observed young magnetars and XDINSs.
young_obs_mask = (age_x_real_obs <= 2) | (age_char_x_obs <= 2)
xdins_obs_mask = ((age_x_real_obs >= 5) | (age_char_x_obs >= 5)) & (
    d_x_real_obs <= 0.5
)

In [ ]:
P_x_young_obs = P_x_obs[young_obs_mask]
Pdot_x_young_obs = Pdot_x_obs[young_obs_mask]
S_x_young_obs = S_x_obs[young_obs_mask]

P_xdins_obs = P_x_obs[xdins_obs_mask]
Pdot_xdins_obs = Pdot_x_obs[xdins_obs_mask]
S_xdins_obs = S_x_obs[xdins_obs_mask]

## Load the simulations

We have simulated 100 populations of neutron stars using the best-parameter values sampled from the inferred posterior distribution. 
- When the entire observed X-ray population is considered (for Figure 3), we use the trained posterior estimator saved at the following path: `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235/round_4`.
- When only the sample of young magnetars and XDINSs is considered for inference (for Figure 7), we use the trained posterior estimator saved at the following path: `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4`.

In [ ]:
root_path = "../../data/paper_results/ronchi_etal_2026/experiments_paper"

# By default, we consider the entire X-ray sample to reproduce Figures 3 and 4.
# To consider the inference results using only young magnetars and XDINSs and
# reproduce Figures 7 and 8, we have to set `use_young_xdins_only` to True.
use_young_xdins_only = True

if use_young_xdins_only:
    simulations_path = f"{root_path}/tsnpe_experiment_1_maps8_res32_youngxdins/simulations_best_params/output_simulations"
else:
    simulations_path = (
        f"{root_path}/tsnpe_experiment_1_maps8_res32/simulations_best_params/output_simulations"
    )

# Number of samples in the parsed directory.
n_sim = len(next(os.walk(simulations_path))[1])

print(n_sim)

In [ ]:
P_radio_sim_list = []
Pdot_radio_sim_list = []

P_pmps_sim_list = []
Pdot_pmps_sim_list = []
S1400_pmps_sim_list = []

P_smps_sim_list = []
Pdot_smps_sim_list = []
S1400_smps_sim_list = []

P_htru_sim_list = []
Pdot_htru_sim_list = []
S1400_htru_sim_list = []

P_x_sim_list = []
Pdot_x_sim_list = []
d_x_sim_list = []
S_x_sim_list = []

P_xdins_sim_list = []
Pdot_xdins_sim_list = []
S_xdins_sim_list = []

P_x_young_sim_list = []
Pdot_x_young_sim_list = []
S_x_young_sim_list = []

n_young_sim_list = []
n_xdins_sim_list = []

br_pmps_list = []
br_smps_list = []
br_htru_list = []
br_x_list = []

nearby_ns = []

In [ ]:
# Load simulation data and save the relevant parameters into lists. In particular,
# we need the spin period P, its derivative Pdot, the radio and X-ray fluxes.

for i in range(n_sim):
    path_to_simulation = f"{simulations_path}/{i:06d}"

    config_json = json.load(
        open(
            pathlib.Path().joinpath(path_to_simulation, "configuration.json"),
        )
    )

    # Skip simulations where the birth rate exceeded the maximum allowed to not bias the birth rate estimate.
    if (
        (config_json["birth_rate_PMPS_at_match"] == 0)
        | (config_json["birth_rate_SMPS_at_match"] == 0)
        | (config_json["birth_rate_HTRU_low_mid_at_match"] == 0)
        | (config_json["birth_rate_xray_realistic_at_match"] == 0)
    ):
        continue

    # Load the `.pkl.gz` files containing the survey results to import.
    df_PMPS_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_PMPS_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_PMPS_sim.head()

    df_SMPS_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_SMPS_results.pkl.gz"
        ),
        compression="gzip",
    )

    df_HTRU_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_HTRU_low_mid_results.pkl.gz"
        ),
        compression="gzip",
    )
    df_x_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_xray_realistic_results.pkl.gz"
        ),
        compression="gzip",
    )

    # Extract relevant quantities.
    P_pmps_sim = df_PMPS_sim["P"]["[s]"].to_numpy()
    Pdot_pmps_sim = df_PMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
    S1400_pmps_sim = df_PMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy() * 1.e3 # convert from Jy to mJy.

    P_smps_sim = df_SMPS_sim["P"]["[s]"].to_numpy()
    Pdot_smps_sim = df_SMPS_sim["P_dot"]["[s s^-1]"].to_numpy()
    S1400_smps_sim = df_SMPS_sim["S_radio_obs_mean"]["[Jy]"].to_numpy() * 1.e3 # convert from Jy to mJy.

    P_htru_sim = df_HTRU_sim["P"]["[s]"].to_numpy()
    Pdot_htru_sim = df_HTRU_sim["P_dot"]["[s s^-1]"].to_numpy()
    S1400_htru_sim = df_HTRU_sim["S_radio_obs_mean"]["[Jy]"].to_numpy() * 1.e3 # convert from Jy to mJy.

    # Remove duplicates, i.e., same pulsars detected with different radio surveys.
    unique_vals, idx = np.unique(
        np.concatenate((P_pmps_sim, P_smps_sim, P_htru_sim)), return_index=True
    )
    P_radio_sim = unique_vals[np.argsort(idx)]
    unique_vals, idx = np.unique(
        np.concatenate((Pdot_pmps_sim, Pdot_smps_sim, Pdot_htru_sim)),
        return_index=True,
    )
    Pdot_radio_sim = unique_vals[np.argsort(idx)]

    P_x_sim = df_x_sim["P"]["[s]"].to_numpy()
    Pdot_x_sim = df_x_sim["P_dot"]["[s s^-1]"].to_numpy()
    S_x_sim = df_x_sim["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy()
    d_x_sim = df_x_sim["dist"]["[kpc]"].to_numpy()
    age_x_sim = df_x_sim["age"]["[yr]"].to_numpy()

    # Filter young magnetars.
    young_mask = age_x_sim <= 2.0e3

    P_x_young_sim = P_x_sim[young_mask]
    Pdot_x_young_sim = Pdot_x_sim[young_mask]
    S_x_young_sim = S_x_sim[young_mask]

    n_young_sim_list.append(len(P_x_sim[young_mask]))

    # Filter X-ray emitting NSs with XDINS-like properties.
    xdins_mask = (d_x_sim <= 0.5) & (age_x_sim >= 1.0e5)

    P_xdins_sim = P_x_sim[xdins_mask]
    Pdot_xdins_sim = Pdot_x_sim[xdins_mask]
    S_xdins_sim = S_x_sim[xdins_mask]

    n_xdins_sim_list.append(len(P_x_sim[xdins_mask]))

    # Append the values to the corresponding lists.
    P_radio_sim_list.append(P_radio_sim)
    Pdot_radio_sim_list.append(Pdot_radio_sim)

    P_pmps_sim_list.append(P_pmps_sim)
    Pdot_pmps_sim_list.append(Pdot_pmps_sim)
    S1400_pmps_sim_list.append(S1400_pmps_sim)

    P_smps_sim_list.append(P_smps_sim)
    Pdot_smps_sim_list.append(Pdot_smps_sim)
    S1400_smps_sim_list.append(S1400_smps_sim)

    P_htru_sim_list.append(P_htru_sim)
    Pdot_htru_sim_list.append(Pdot_htru_sim)
    S1400_htru_sim_list.append(S1400_htru_sim)

    P_x_sim_list.append(P_x_sim)
    Pdot_x_sim_list.append(Pdot_x_sim)
    S_x_sim_list.append(S_x_sim)

    P_x_young_sim_list.append(P_x_young_sim)
    Pdot_x_young_sim_list.append(Pdot_x_young_sim)
    S_x_young_sim_list.append(S_x_young_sim)

    P_xdins_sim_list.append(P_xdins_sim)
    Pdot_xdins_sim_list.append(Pdot_xdins_sim)
    S_xdins_sim_list.append(S_xdins_sim)

    # Extract birth rate information.
    br_pmps_list.append(config_json["birth_rate_PMPS_at_match"])
    br_smps_list.append(config_json["birth_rate_SMPS_at_match"])
    br_htru_list.append(config_json["birth_rate_HTRU_low_mid_at_match"])
    br_x_list.append(config_json["birth_rate_xray_realistic_at_match"])

In [ ]:
# Flatten arrays.
P_radio_sim_all = np.concatenate(P_radio_sim_list)
Pdot_radio_sim_all = np.concatenate(Pdot_radio_sim_list)

P_pmps_sim_all = np.concatenate(P_pmps_sim_list)
Pdot_pmps_sim_all = np.concatenate(Pdot_pmps_sim_list)
S1400_pmps_sim_all = np.concatenate(S1400_pmps_sim_list)

P_smps_sim_all = np.concatenate(P_smps_sim_list)
Pdot_smps_sim_all = np.concatenate(Pdot_smps_sim_list)
S1400_smps_sim_all = np.concatenate(S1400_smps_sim_list)

P_htru_sim_all = np.concatenate(P_htru_sim_list)
Pdot_htru_sim_all = np.concatenate(Pdot_htru_sim_list)
S1400_htru_sim_all = np.concatenate(S1400_htru_sim_list)

P_x_sim_all = np.concatenate(P_x_sim_list)
Pdot_x_sim_all = np.concatenate(Pdot_x_sim_list)
S_x_sim_all = np.concatenate(S_x_sim_list)

P_x_young_sim_all = np.concatenate(P_x_young_sim_list)
Pdot_x_young_sim_all = np.concatenate(Pdot_x_young_sim_list)
S_x_young_sim_all = np.concatenate(S_x_young_sim_list)

P_xdins_sim_all = np.concatenate(P_xdins_sim_list)
Pdot_xdins_sim_all = np.concatenate(Pdot_xdins_sim_list)
S_xdins_sim_all = np.concatenate(S_xdins_sim_list)

## Plot the $P-\dot{P}$ diagram

Compare the distribution of the simulated neutron star population with the observed one in the $P-\dot{P}$ diagram.
If `use_young_xdins_only = False`, this reproduces Figure 3 in Ronchi et al. (2026).
If `use_young_xdins_only = True`, this reproduces Figure 7 in Ronchi et al. (2026).

In [ ]:
def density_contour(x, y, ax, **kwargs):
    """
    Helper function to make a density contour plot from a point cloud.
    """
    xy = np.vstack([x, y])
    kde = gaussian_kde(xy)
    xi, yi = np.meshgrid(
        np.linspace(np.min(x), np.max(x), 200),
        np.linspace(np.min(y), np.max(y), 200),
    )
    zi = kde(np.vstack([xi.flatten(), yi.flatten()]))
    zi = zi.reshape(xi.shape)

    return ax.contour(10**xi, 10**yi, zi, **kwargs)

In [ ]:
P_limits = [0.005, 200]
Pdot_limits = [1.0e-22, 1.0e-7]

# Create a meshgrid to evaluate the contour levels of constant magnetic field,
# characteristic age and spin-down power.

log_P_edges = np.linspace(np.log10(P_limits[0]), np.log10(P_limits[1]), 51)
log_P_centers = 0.5 * (log_P_edges[1:] + log_P_edges[:-1])
P_edges = 10**log_P_edges
P_centers = 10**log_P_centers

log_Pdot_edges = np.linspace(
    np.log10(Pdot_limits[0]), np.log10(Pdot_limits[1]), 51
)
log_Pdot_centers = 0.5 * (log_Pdot_edges[1:] + log_Pdot_edges[:-1])
Pdot_edges = 10**log_Pdot_edges
Pdot_centers = 10**log_Pdot_centers

P_grid, Pdot_grid = np.meshgrid(P_centers, Pdot_centers, indexing="ij")

B_timing = B_from_timing(P_grid, Pdot_grid)
tage_timing = tage_from_timing(P_grid, Pdot_grid)
Edot_timing = Edot_from_timing(P_grid, Pdot_grid)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(P_limits[0], P_limits[1])
ax.set_ylim(1.0e-22, Pdot_limits[1])
ax.set_xlabel(r"Spin period [s]")
ax.set_ylabel(r"Spin period derivative")
ax.xaxis.set_major_formatter(formatter)

ax.scatter(
    P_pmps_obs,
    Pdot_pmps_obs,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=15,
    alpha=1,
    rasterized=True,
    label=r"Observed isolated radio pulsars",
    zorder=3,
)

ax.scatter(
    P_smps_obs,
    Pdot_smps_obs,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=15,
    alpha=1,
    rasterized=True,
    zorder=3,
)

ax.scatter(
    P_htru_obs,
    Pdot_htru_obs,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=15,
    alpha=1,
    rasterized=True,
    zorder=3,
)

ax.scatter(
    P_x_obs,
    Pdot_x_obs,
    linestyle="None",
    marker="X",
    facecolors="tab:purple",
    edgecolors="None",
    s=200,
    alpha=1,
    rasterized=True,
    label=r"Observed thermally emitting NSs",
    zorder=3,
)

ax.scatter(
    P_x_young_obs,
    Pdot_x_young_obs,
    linestyle="None",
    marker="X",
    facecolors="#E6A0C4",
    edgecolors="None",
    s=200,
    alpha=1,
    rasterized=True,
    label=r"Observed thermally emitting NSs (age < 2 kyr)",
    zorder=3,
)

ax.scatter(
    P_xdins_obs,
    Pdot_xdins_obs,
    linestyle="None",
    marker="X",
    facecolors="tab:orange",
    edgecolors="tab:orange",
    s=200,
    alpha=1,
    rasterized=True,
    label=r"Observed XDINSs",
    zorder=4,
)

contour_B = ax.contour(
    P_grid,
    Pdot_grid,
    np.log10(B_timing),
    levels=np.array([10.0, 12.0, 14.0]),
    colors="darkgray",
    linestyles="solid",
    # interpolation='none'
)
fmt = {}
strs = ["$10^{10}$ G", "$10^{12}$ G", "$10^{14}$ G"]
for l, s in zip(contour_B.levels, strs):
    fmt[l] = rf"{s}"
manual_locations = [(0.02, 1e-16), (0.02, 1e-12), (0.02, 1e-9)]
ax.clabel(
    contour_B,
    contour_B.levels,
    inline=True,
    manual=manual_locations,
    fmt=fmt,
    fontsize=20,
    colors="black",
)

contour_Edot = ax.contour(
    P_grid,
    Pdot_grid,
    np.log10(Edot_timing),
    levels=np.array([28.0, 31.0, 34.0, 37.0, 40.0]),
    colors="darkgray",
    linestyles="dashed",
    # interpolation='none'
)
fmt = {}
strs = [
    "$10^{28}$ erg s$^{-1}$",
    "$10^{31}$ erg s$^{-1}$",
    "$10^{34}$ erg s$^{-1}$",
    "$10^{37}$ erg s$^{-1}$",
    "$10^{40}$ erg s$^{-1}$",
]
for l, s in zip(contour_Edot.levels, strs):
    fmt[l] = rf"{s}"
manual_locations = [
    (0.2, 1e-8),
    (2, 1e-8),
    (20, 1e-8),
    (70, 1e-11),
    (30, 1e-16),
]
ax.clabel(
    contour_Edot,
    contour_Edot.levels,
    inline=True,
    manual=manual_locations,
    fmt=fmt,
    fontsize=20,
    colors="black",
)
"""
contour_tage = ax.contour(
    P_grid, 
    Pdot_grid,
    np.log10(tage_timing),
    levels = np.array([2.,4.,6.,8.,10.]), 
    colors='gray',
    linestyles='dashed'
    #interpolation='none'
)
fmt = {}
strs = ['$10^{2}$ yr', '$10^{4}$ yr', '$10^{6}$ yr', '$10^{8}$ yr', '$10^{10}$ yr']
for l,s in zip( contour_tage.levels, strs ):
    fmt[l] = rf"{s}"
manual_locations = [(5e-4, 1e-13), (5e-4, 1e-15), (5e-4, 1e-17), (5e-4, 1e-19), (5e-4, 1e-21)]
ax.clabel(contour_tage, contour_tage.levels, inline=True, manual=manual_locations, fmt=fmt, fontsize=20, colors='black')
"""

# Example: replace scatter with contours
# density_contour(np.log10(P_pk_sim), np.log10(Pdot_pk_sim), ax, colors="tab:red", levels=5, linewidths=2)
# density_contour(np.log10(P_sw_sim), np.log10(Pdot_sw_sim), ax, colors="tab:blue", levels=5, linewidths=2)
# density_contour(np.log10(P_htru_sim), np.log10(Pdot_htru_sim), ax, colors="tab:green", levels=5, linewidths=2)
density_contour(
    np.log10(P_x_sim_all),
    np.log10(Pdot_x_sim_all),
    ax,
    colors="#d6c2ea",
    levels=20,
    linewidths=3,
)
density_contour(
    np.log10(P_radio_sim_all),
    np.log10(Pdot_radio_sim_all),
    ax,
    colors="darkgray",
    levels=20,
    linewidths=3,
)
density_contour(
    np.log10(P_xdins_sim_all),
    np.log10(Pdot_xdins_sim_all),
    ax,
    colors="#ffb36b",
    levels=5,
    linewidths=3,
)
density_contour(
    np.log10(P_x_young_sim_all),
    np.log10(Pdot_x_young_sim_all),
    ax,
    colors="#FFB3D9",
    levels=5,
    linewidths=3,
)

# --- Collect existing legend entries (from scatter plots, etc.) ---
handles, labels = ax.get_legend_handles_labels()

# --- Final legend with all items ---
# Create proxy handles for legend
kde_x_proxy = mlines.Line2D(
    [], [], color="#d6c2ea", linewidth=2, label="X-ray KDE"
)
kde_radio_proxy = mlines.Line2D(
    [], [], color="darkgray", linewidth=2, label="Radio KDE"
)
kde_xdins_proxy = mlines.Line2D(
    [], [], color="#ffb36b", linewidth=2, label="XDINSs KDE"
)
kde_xyoung_proxy = mlines.Line2D(
    [], [], color="#FFB3D9", linewidth=2, label="X-ray young KDE"
)

# --- Add the KDE proxies ---
handles.extend(
    [kde_radio_proxy, kde_x_proxy, kde_xdins_proxy, kde_xyoung_proxy]
)
labels.extend(
    [
        "Simulated radio KDE",
        "Simulated X-ray KDE",
        "Simulated XDINS KDE",
        "Simulated X-ray young KDE",
    ]
)

ax.legend(handles, labels, frameon=True, loc=4, prop={"size": 14})


if use_young_xdins_only:
    fig.savefig("plots/P-Pdot_sim_vs_obs_youngxdins.png", bbox_inches="tight")
else:
    fig.savefig("plots/P-Pdot_sim_vs_obs.png", bbox_inches="tight")

plt.show()

## Plot relation between X-ray fluxes and timing properties

Plot X-ray flux vs. spin period for the X-ray sample.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_xscale("log")
ax.set_yscale("log")

ax.plot(
    P_x_obs,
    S_x_obs,
    marker="X",
    color="tab:purple",
    linestyle="None",
    markersize=15,
    alpha=1,
    rasterized=True,
    label="Observed X-ray",
    zorder=3,
)

ax.plot(
    P_xdins_obs,
    S_xdins_obs,
    marker="X",
    color="tab:orange",
    linestyle="None",
    markersize=15,
    alpha=1,
    rasterized=True,
    label="Observed XDINS",
    zorder=3,
)

ax.plot(
    P_x_young_obs,
    S_x_young_obs,
    marker="X",
    color="#E6A0C4",
    linestyle="None",
    markersize=15,
    alpha=1,
    rasterized=True,
    label="Observed X-ray (age < 2 kyr)",
    zorder=3,
)

density_contour(
    np.log10(P_x_sim_all),
    np.log10(S_x_sim_all),
    ax,
    colors="#d6c2ea",
    levels=20,
    linewidths=3,
)
density_contour(
    np.log10(P_xdins_sim_all),
    np.log10(S_xdins_sim_all),
    ax,
    colors="#ffb36b",
    levels=5,
    linewidths=3,
)
density_contour(
    np.log10(P_x_young_sim_all),
    np.log10(S_x_young_sim_all),
    ax,
    colors="#FFB3D9",
    levels=5,
    linewidths=3,
)

# Collect existing legend entries (from scatter plots, etc.).
handles, labels = ax.get_legend_handles_labels()

# Final legend with all items.
# Create proxy handles for legend.
kde_x_proxy = mlines.Line2D(
    [], [], color="#d6c2ea", linewidth=2, label="X-ray KDE"
)
kde_xdins_proxy = mlines.Line2D(
    [], [], color="#ffb36b", linewidth=2, label="XDINS KDE"
)
kde_xyoung_proxy = mlines.Line2D(
    [], [], color="#FFB3D9", linewidth=2, label="X-ray young KDE"
)

# Add the KDE proxies.
handles.extend([kde_x_proxy, kde_xdins_proxy, kde_xyoung_proxy])
labels.extend(
    ["Simulated X-ray KDE", "Simulated XDINS KDE", "Simulated X-ray young KDE"]
)

ax.legend(handles, labels, frameon=True, loc=4, prop={"size": 14})

plt.grid()

Plot X-ray flux vs. spin period derivative for the X-ray sample.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

ax.set_xlabel(r"$\dot{P}$")
ax.set_ylabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
ax.set_xscale("log")
ax.set_yscale("log")

ax.plot(
    Pdot_x_obs,
    S_x_obs,
    marker="X",
    color="tab:purple",
    linestyle="None",
    markersize=15,
    alpha=1,
    rasterized=True,
    label="Observed X-ray",
    zorder=3,
)

ax.plot(
    Pdot_xdins_obs,
    S_xdins_obs,
    marker="X",
    color="tab:orange",
    linestyle="None",
    markersize=15,
    alpha=1,
    rasterized=True,
    label="Observed X-ray",
    zorder=3,
)

ax.plot(
    Pdot_x_young_obs,
    S_x_young_obs,
    marker="X",
    color="#E6A0C4",
    linestyle="None",
    markersize=15,
    alpha=1,
    rasterized=True,
    label="Observed X-ray (age < 2 kyr)",
    zorder=3,
)

density_contour(
    np.log10(Pdot_x_sim_all),
    np.log10(S_x_sim_all),
    ax,
    colors="#d6c2ea",
    levels=20,
    linewidths=3,
)
density_contour(
    np.log10(Pdot_xdins_sim_all),
    np.log10(S_xdins_sim_all),
    ax,
    colors="#ffb36b",
    levels=5,
    linewidths=3,
)
density_contour(
    np.log10(Pdot_x_young_sim_all),
    np.log10(S_x_young_sim_all),
    ax,
    colors="#FFB3D9",
    levels=5,
    linewidths=3,
)

# Collect existing legend entries (from scatter plots, etc.).
handles, labels = ax.get_legend_handles_labels()

# Final legend with all items.
# Create proxy handles for legend.
kde_x_proxy = mlines.Line2D(
    [], [], color="#d6c2ea", linewidth=2, label="X-ray KDE"
)
kde_xdins_proxy = mlines.Line2D(
    [], [], color="#ffb36b", linewidth=2, label="XDINS KDE"
)
kde_xyoung_proxy = mlines.Line2D(
    [], [], color="#FFB3D9", linewidth=2, label="X-ray young KDE"
)

# Add the KDE proxies.
handles.extend([kde_x_proxy, kde_xdins_proxy, kde_xyoung_proxy])
labels.extend(
    ["Simulated X-ray KDE", "Simulated XDINS KDE", "Simulated X-ray young KDE"]
)

ax.legend(handles, labels, frameon=True, loc=4, prop={"size": 14})

plt.grid()

## Radio flux distribution
Compare the distribution of radio fluxes between simulations and observations.

In [ ]:
colors = ["#FFAC1C", "dodgerblue", "#440154"]

In [ ]:
# New X-axis values.
x = np.linspace(-2, 3.5, 1000)

kde_pmps_sim_list = []
kde_smps_sim_list = []
kde_htru_sim_list = []

for S in S1400_pmps_sim_list:
    # Estimate the PDF using a Gaussian kernel.
    kde_pmps_sim = gaussian_kde(np.log10(S))
    kde_pmps_sim_list.append(kde_pmps_sim(x))

for S in S1400_smps_sim_list:
    # Estimate the PDF using a Gaussian kernel.
    kde_smps_sim = gaussian_kde(np.log10(S))
    kde_smps_sim_list.append(kde_smps_sim(x))

for S in S1400_htru_sim_list:
    # Estimate the PDF using a Gaussian kernel.
    kde_htru_sim = gaussian_kde(np.log10(S))
    kde_htru_sim_list.append(kde_htru_sim(x))

kde_pmps_sim_list = np.vstack(kde_pmps_sim_list)
kde_smps_sim_list = np.vstack(kde_smps_sim_list)
kde_htru_sim_list = np.vstack(kde_htru_sim_list)

In [ ]:
# Compute the percentiles for the median radio flux distribution and the 1-sigma and 3-sigma uncertainties.
p01_kde_pmps_sim = np.nanpercentile(kde_pmps_sim_list, 0.15, axis=0)
p16_kde_pmps_sim = np.nanpercentile(kde_pmps_sim_list, 15.85, axis=0)
p50_kde_pmps_sim = np.nanpercentile(kde_pmps_sim_list, 50, axis=0)  # median
p84_kde_pmps_sim = np.nanpercentile(kde_pmps_sim_list, 84.15, axis=0)
p99_kde_pmps_sim = np.nanpercentile(kde_pmps_sim_list, 99.85, axis=0)

p01_kde_smps_sim = np.nanpercentile(kde_smps_sim_list, 0.15, axis=0)
p16_kde_smps_sim = np.nanpercentile(kde_smps_sim_list, 15.85, axis=0)
p50_kde_smps_sim = np.nanpercentile(kde_smps_sim_list, 50, axis=0)  # median
p84_kde_smps_sim = np.nanpercentile(kde_smps_sim_list, 84.15, axis=0)
p99_kde_smps_sim = np.nanpercentile(kde_smps_sim_list, 99.85, axis=0)

p01_kde_htru_sim = np.nanpercentile(kde_htru_sim_list, 0.15, axis=0)
p16_kde_htru_sim = np.nanpercentile(kde_htru_sim_list, 15.85, axis=0)
p50_kde_htru_sim = np.nanpercentile(kde_htru_sim_list, 50, axis=0)  # median
p84_kde_htru_sim = np.nanpercentile(kde_htru_sim_list, 84.15, axis=0)
p99_kde_htru_sim = np.nanpercentile(kde_htru_sim_list, 99.85, axis=0)

In [ ]:
# Remove NaNs and estimate the PDF using a Gaussian kernel.
S1400_pmps_obs = S1400_pmps_obs[~np.isnan(S1400_pmps_obs)]
kde_pmps_obs = gaussian_kde(np.log10(S1400_pmps_obs))

S1400_smps_obs = S1400_smps_obs[~np.isnan(S1400_smps_obs)]
kde_smps_obs = gaussian_kde(np.log10(S1400_smps_obs))

S1400_htru_obs = S1400_htru_obs[~np.isnan(S1400_htru_obs)]
kde_htru_obs = gaussian_kde(np.log10(S1400_htru_obs))

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

# KDE density plots simulated.
ax.plot(
    x,
    p50_kde_pmps_sim,
    linewidth=5,
    linestyle="--",
    label=r"KDE simulated PMPS",
    color=colors[0],
)

ax.fill_between(
    x, p16_kde_pmps_sim, p84_kde_pmps_sim, alpha=0.4, color=colors[0]
)

ax.plot(
    x,
    p50_kde_smps_sim,
    linewidth=5,
    linestyle="--",
    label=r"KDE simulated SMPS",
    color=colors[1],
)
ax.fill_between(
    x, p16_kde_smps_sim, p84_kde_smps_sim, alpha=0.4, color=colors[1]
)

ax.plot(
    x,
    p50_kde_htru_sim,
    linewidth=5,
    linestyle="--",
    label=r"KDE simulated HTRU",
    color=colors[2],
)
ax.fill_between(
    x, p16_kde_htru_sim, p84_kde_htru_sim, alpha=0.4, color=colors[2]
)


# KDE density plots ATNF populations.
ax.plot(
    x,
    kde_pmps_obs(x),
    color=colors[0],
    alpha=0.5,
    linewidth=5,
    label=r"KDE observed PMPS",
)
ax.plot(
    x,
    kde_smps_obs(x),
    color=colors[1],
    alpha=0.5,
    linewidth=5,
    label=r"KDE observed SMPS",
)
ax.plot(
    x,
    kde_htru_obs(x),
    color=colors[2],
    alpha=0.5,
    linewidth=5,
    label=r"KDE observed HTRU",
)

ax.set_xlim(-2, 3.5)
ax.set_ylim(0.0, 1.0)
ax.set_xlabel(r"Mean flux density log$_{10}$ $S_{{\rm mean}, 1400}$ [mJy]")
ax.set_ylabel(r"Normalized pulsar count")
ax.legend(frameon=True, loc="best")

plt.tight_layout()

plt.show()

## Birth rate estimates
Estimate birth rates for each simulated survey (see Eqs. 20 and 21 in Ronchi et al. 2026).

In [ ]:
br_pmps_mean = np.mean(br_pmps_list[br_pmps_list != 0])
br_smps_mean = np.mean(br_smps_list[br_smps_list != 0])
br_htru_mean = np.mean(br_htru_list[br_htru_list != 0])
br_x_mean = np.mean(br_x_list[br_x_list != 0])

br_pmps_std = np.std(br_pmps_list[br_pmps_list != 0])
br_smps_std = np.std(br_smps_list[br_smps_list != 0])
br_htru_std = np.std(br_htru_list[br_htru_list != 0])
br_xray_std = np.std(br_x_list[br_x_list != 0])

print(f"birth rate PMPS: {np.mean(br_pmps_list)} +- {np.std(br_pmps_list)}")
print(f"birth rate SMPS: {np.mean(br_smps_list)} +- {np.std(br_smps_list)}")
print(f"birth rate HTRU: {np.mean(br_htru_list)} +- {np.std(br_htru_list)}")
print(f"birth rate X-ray: {np.mean(br_x_list)} +- {np.std(br_x_list)}")

Estimate the number of young magnetars and XDINS-like neutron stars in the simulations.

In [ ]:
print(
    f"number of XDINS-like: {np.mean(n_xdins_sim_list)} +- {np.std(n_xdins_sim_list)}"
)

In [ ]:
print(
    f"number of young magnetars: {np.mean(n_young_sim_list)} +- {np.std(n_young_sim_list)}"
)